# Mixed-effects models of pre-to-post accuracy

Estimate differential changes in condition and triage accuracy across study arms.

## Load packages

In [ ]:
# Pakete laden — Installation war einmalig nötig (install.packages(...)).
# Diese Zelle jedes Mal ausführen; erneut installieren nur bei fehlendem Paket:
#   install.packages(c("readxl", "dplyr", "lme4", "emmeans"))
# Word-Export (letzte Zelle): install.packages(c("officer", "flextable"))

library(readxl)
library(dplyr)
library(lme4)
library(emmeans)

## Prepare accuracy data and model helpers

In [ ]:
ROOT <- normalizePath("..", winslash = "/")
PATH <- file.path(ROOT, "data", "Psy_Case_Vignette_Data_Prolific_analysis.xlsx")
VIGNETTES <- c("A", "B", "C", "D", "E")
ARM_REF <- "4"
COUNTRY_COL <- "Country of residence"
SAMPLES <- c("United Kingdom", "United States")
ADJ_COVARIATES <- c("Age", "Sex", "ai_frequency", "ai_literacy_sum")
AILI_ITEMS <- paste0("aili_", sprintf("%02d", 1:12))
REVERSE_AILI_ITEMS <- c("aili_02", "aili_05", "aili_11")

df <- read_excel(PATH)

recode_aili_col <- function(col_name) {
  values <- as.numeric(df[[col_name]])
  if (col_name %in% REVERSE_AILI_ITEMS) {
    values <- 8 - values
  }
  values
}

aili_recoded <- as.data.frame(lapply(AILI_ITEMS, recode_aili_col), stringsAsFactors = FALSE)
names(aili_recoded) <- AILI_ITEMS
df$ai_literacy_sum <- apply(aili_recoded, 1, function(row) {
  if (any(is.na(row))) NA_real_ else sum(row)
})

make_long_outcome <- function(data, pre_cols, post_cols) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    bind_rows(
      data %>%
        transmute(
          prolific_id = as.factor(prolific_id),
          vignette = factor(VIGNETTES[i], levels = VIGNETTES),
          study_arm = as.character(study_arm),
          country = as.character(.data[[COUNTRY_COL]]),
          Age = Age,
          Sex = as.character(Sex),
          ai_frequency = as.numeric(ai_frequency),
          ai_literacy_sum = as.numeric(ai_literacy_sum),
          post = 0L,
          correct = .data[[pre_cols[i]]]
        ),
      data %>%
        transmute(
          prolific_id = as.factor(prolific_id),
          vignette = factor(VIGNETTES[i], levels = VIGNETTES),
          study_arm = as.character(study_arm),
          country = as.character(.data[[COUNTRY_COL]]),
          Age = Age,
          Sex = as.character(Sex),
          ai_frequency = as.numeric(ai_frequency),
          ai_literacy_sum = as.numeric(ai_literacy_sum),
          post = 1L,
          correct = .data[[post_cols[i]]]
        )
    )
  })) %>%
    filter(
      !is.na(correct), !is.na(study_arm), !is.na(prolific_id),
      !is.na(country), !is.na(Age), !is.na(Sex)
    ) %>%
    mutate(
      correct = as.integer(correct),
      post = as.integer(post),
      study_arm = relevel(factor(study_arm), ref = ARM_REF),
      Sex = relevel(factor(Sex), ref = "Female")
    )
}

print_post_terms <- function(model, label) {
  cat("\n", label, "\n", sep = "")
  cat(
    "N participants:", n_distinct(model@frame$prolific_id),
    "; N observations:", nrow(model@frame), "\n\n"
  )
  print(summary(model))
  coef_mat <- coef(summary(model))
  cat("\nPre→post odds ratios (post = control arm; interactions = difference vs control):\n")
  post_rows <- grep("^post", rownames(coef_mat), value = TRUE)
  for (nm in post_rows) {
    b <- coef_mat[nm, "Estimate"]
    se <- coef_mat[nm, "Std. Error"]
    p <- coef_mat[nm, "Pr(>|z|)"]
    cat(sprintf(
      "  %s: OR = %.3f [%.3f, %.3f], p = %.4f\n",
      nm, exp(b), exp(b - 1.96 * se), exp(b + 1.96 * se), p
    ))
  }
  invisible(model)
}

print_post_study_arm_omnibus <- function(full_model, label, covariates = character()) {
  cov <- if (length(covariates)) paste("+", paste(covariates, collapse = " + ")) else ""
  reduced_formula <- as.formula(
    paste("correct ~ post + study_arm", cov, "+ (1 | prolific_id) + (1 | vignette)")
  )
  cat("\n", label, "— omnibus post × study_arm interaction (LRT, ML)\n", sep = "")
  reduced_model <- update(full_model, formula = reduced_formula)
  cmp <- anova(reduced_model, full_model)
  print(cmp)
  invisible(cmp)
}

GLMER_CTRL <- glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))

sample_label <- function(country) {
  if (country == "United Kingdom") "UK" else "US"
}

fit_accuracy_by_sample <- function(long_data, outcome_label) {
  for (country in SAMPLES) {
    sub <- long_data %>% filter(country == .env$country)
    label <- paste0(outcome_label, " (", sample_label(country), ", unadjusted)")
    cat("\n", strrep("=", 88), "\n", sep = "")
    cat(label, "\n")
    cat(strrep("=", 88), "\n", sep = "")
    full_model <- glmer(
      correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
      data = sub,
      family = binomial,
      control = GLMER_CTRL
    )
    print_post_terms(full_model, label)
    print_post_study_arm_omnibus(full_model, label)
  }
  invisible(NULL)
}

PRE_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_pre_condition_correct")
POST_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_post_condition_correct")
PRE_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_pre_nextstep_correct")
POST_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_post_nextstep_correct")

long_condition <- make_long_outcome(df, PRE_CONDITION_COLS, POST_CONDITION_COLS)
long_triage <- make_long_outcome(df, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS)

long_condition_adj <- long_condition %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))
long_triage_adj <- long_triage %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))

cat("long_condition:", nrow(long_condition), "rows,", n_distinct(long_condition$prolific_id), "participants\n")
cat("long_triage:   ", nrow(long_triage), "rows,", n_distinct(long_triage$prolific_id), "participants\n")
cat("long_condition_adj:", nrow(long_condition_adj), "rows,", n_distinct(long_condition_adj$prolific_id), "participants\n")
cat("long_triage_adj:   ", nrow(long_triage_adj), "rows,", n_distinct(long_triage_adj$prolific_id), "participants\n")

### Fit the unadjusted condition model

In [ ]:
condition_unadj <- glmer(
  correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_condition,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(condition_unadj, "Model 1 — Condition diagnosis (unadjusted)")
print_post_study_arm_omnibus(condition_unadj, "Model 1 — Condition diagnosis (unadjusted)")

### Fit the adjusted condition model

In [ ]:
condition_adj <- glmer(
  correct ~ post * study_arm + Age + Sex + ai_frequency + ai_literacy_sum +
    (1 | prolific_id) + (1 | vignette),
  data = long_condition_adj,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(
  condition_adj,
  "Model 2 — Condition diagnosis (adjusted for Age, Sex, AI frequency, AI literacy)"
)
print_post_study_arm_omnibus(
  condition_adj,
  "Model 2 — Condition diagnosis (adjusted for Age, Sex, AI frequency, AI literacy)",
  covariates = ADJ_COVARIATES
)

### Fit the unadjusted triage model

In [ ]:
triage_unadj <- glmer(
  correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_triage,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(triage_unadj, "Model 3 — Triage / next step (unadjusted)")
print_post_study_arm_omnibus(triage_unadj, "Model 3 — Triage / next step (unadjusted)")

### Fit the adjusted triage model

In [ ]:
triage_adj <- glmer(
  correct ~ post * study_arm + Age + Sex + ai_frequency + ai_literacy_sum +
    (1 | prolific_id) + (1 | vignette),
  data = long_triage_adj,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(
  triage_adj,
  "Model 4 — Triage / next step (adjusted for Age, Sex, AI frequency, AI literacy)"
)
print_post_study_arm_omnibus(
  triage_adj,
  "Model 4 — Triage / next step (adjusted for Age, Sex, AI frequency, AI literacy)",
  covariates = ADJ_COVARIATES
)

## Fit country-stratified accuracy models

In [ ]:
for (country in SAMPLES) {
  n_part <- long_condition %>%
    filter(country == .env$country) %>%
    distinct(prolific_id) %>%
    nrow()
  cat(sample_label(country), "participants:", n_part, "\n")
}

fit_accuracy_by_sample(long_condition, "Condition diagnosis")
fit_accuracy_by_sample(long_triage, "Triage / next step")

## Prepare transcript-verification sensitivity samples

SA1 excludes LLM participants with any category-4 consultation. SA2 additionally removes category-2 or category-3 LLM consultations; control observations remain unchanged.

In [ ]:
LLM_ARM_CODES <- c("1", "2", "3")
VALIDATION_COLS <- paste0("vig_", VIGNETTES, "_post_transcript_validation")

excluded_prolific_ids <- function(data, validation_categories) {
  sub <- data %>% filter(as.character(study_arm) %in% LLM_ARM_CODES)
  vals <- sub %>% select(all_of(VALIDATION_COLS))
  has_flag <- apply(vals, 1, function(row) {
    any(as.numeric(row) %in% validation_categories, na.rm = TRUE)
  })
  sub$prolific_id[has_flag]
}

filter_by_validation <- function(data, validation_categories, label) {
  exclude_ids <- excluded_prolific_ids(data, validation_categories)
  cat(
    label, "— excluding", length(exclude_ids),
    "LLM participant(s) with validation in {",
    paste(validation_categories, collapse = ", "), "}\n",
    sep = ""
  )
  filtered <- data %>% filter(!prolific_id %in% exclude_ids)
  cat("Remaining participants:", nrow(filtered), "\n\n")
  filtered
}

vignette_validation_long <- function(data) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    data %>%
      transmute(
        prolific_id = as.factor(prolific_id),
        vignette = factor(VIGNETTES[i], levels = VIGNETTES),
        validation = as.numeric(.data[[VALIDATION_COLS[i]]])
      )
  }))
}

drop_validation_vignettes <- function(long_data, data_wide, validation_categories, label) {
  val_long <- vignette_validation_long(data_wide)
  merged <- long_data %>% left_join(val_long, by = c("prolific_id", "vignette"))
  n_before <- nrow(merged)
  filtered <- merged %>%
    filter(is.na(validation) | !(validation %in% validation_categories)) %>%
    select(-validation)
  cat(
    label, "— excluding", n_before - nrow(filtered),
    "observation(s) from vignettes with validation in {",
    paste(validation_categories, collapse = ", "), "}\n\n",
    sep = ""
  )
  filtered
}

fit_unadjusted_accuracy <- function(long_data, label) {
  model <- glmer(
    correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
    data = long_data,
    family = binomial,
    control = GLMER_CTRL
  )
  print_post_terms(model, label)
  print_post_study_arm_omnibus(model, label)
  invisible(model)
}

df_sa1 <- filter_by_validation(df, 4, "SA1")
df_sa2_participants <- filter_by_validation(df, 4, "SA2 participant exclusion")

long_condition_sa1 <- make_long_outcome(df_sa1, PRE_CONDITION_COLS, POST_CONDITION_COLS)
long_triage_sa1 <- make_long_outcome(df_sa1, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS)

long_condition_sa2 <- make_long_outcome(df_sa2_participants, PRE_CONDITION_COLS, POST_CONDITION_COLS) %>%
  drop_validation_vignettes(df_sa2_participants, c(2, 3), "SA2 vignette exclusion (condition)")
long_triage_sa2 <- make_long_outcome(df_sa2_participants, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS) %>%
  drop_validation_vignettes(df_sa2_participants, c(2, 3), "SA2 vignette exclusion (triage)")

cat("SA1 long_condition:", nrow(long_condition_sa1), "rows,", n_distinct(long_condition_sa1$prolific_id), "participants\n")
cat("SA1 long_triage:   ", nrow(long_triage_sa1), "rows,", n_distinct(long_triage_sa1$prolific_id), "participants\n")
cat("SA2 long_condition:", nrow(long_condition_sa2), "rows,", n_distinct(long_condition_sa2$prolific_id), "participants\n")
cat("SA2 long_triage:   ", nrow(long_triage_sa2), "rows,", n_distinct(long_triage_sa2$prolific_id), "participants\n")

### Fit SA1 accuracy models

In [ ]:
cat(strrep("=", 88), "\n", sep = "")
cat("SA1 — Condition diagnosis (unadjusted; exclude validation category 4)\n")
cat(strrep("=", 88), "\n", sep = "")
condition_sa1 <- fit_unadjusted_accuracy(
  long_condition_sa1,
  "SA1 — Condition diagnosis (unadjusted; exclude validation category 4)"
)

cat(strrep("=", 88), "\n", sep = "")
cat("SA1 — Triage / next step (unadjusted; exclude validation category 4)\n")
cat(strrep("=", 88), "\n", sep = "")
triage_sa1 <- fit_unadjusted_accuracy(
  long_triage_sa1,
  "SA1 — Triage / next step (unadjusted; exclude validation category 4)"
)

### Fit SA2 accuracy models

In [ ]:
cat(strrep("=", 88), "\n", sep = "")
cat("SA2 — Condition diagnosis (unadjusted; exclude category 4 participants, categories 2–3 vignettes)\n")
cat(strrep("=", 88), "\n", sep = "")
condition_sa2 <- fit_unadjusted_accuracy(
  long_condition_sa2,
  "SA2 — Condition diagnosis (unadjusted; exclude category 4 participants, categories 2–3 vignettes)"
)

cat(strrep("=", 88), "\n", sep = "")
cat("SA2 — Triage / next step (unadjusted; exclude category 4 participants, categories 2–3 vignettes)\n")
cat(strrep("=", 88), "\n", sep = "")
triage_sa2 <- fit_unadjusted_accuracy(
  long_triage_sa2,
  "SA2 — Triage / next step (unadjusted; exclude category 4 participants, categories 2–3 vignettes)"
)